# COMP0178 — Lab 5: Protecting Your Database and Passwords

Source: [Lab 5 PDF](Lab5.pdf), 31 PDF pages. This notebook follows the tutorial topics and adds current safe implementation guidance where the slides use outdated database-security examples.

**Main idea:** web-form input is untrusted. Validate it on the server, keep authentication secrets safe, separate SQL code from user data, escape output for HTML, and use server logs to investigate failures or unusual activity.

The examples are PHP/MySQL because that is the tutorial stack. They are study examples; adapt connection/table/column names to your own project.


## 1. Validate data submitted through a form (PDF pages 2–5)

A browser form is not a security boundary. A person can send an HTTP request without using your form, change submitted values, or disable client-side checks. **Repeat important validation on the server before using input.**

Useful PHP checks shown in the tutorial:

- `isset($_POST['user_name'])`: was the field supplied?
- `trim($value)`: remove whitespace at the ends.
- `empty($value)`: test whether a value is empty-like (including `"0"`, so consider whether that fits your field).
- `is_numeric($value)`: whether a value is numeric.
- `preg_match(...)`: check a format with a regular expression.
- `filter_var(...)`: apply a built-in validation or sanitization filter, such as email validation.

A basic safe pattern is:

```php
if (!isset($_POST['user_name'])) {
    $error = 'Please enter a name.';
} else {
    $userName = trim($_POST['user_name']);
    if ($userName === '') {
        $error = 'Name cannot be blank.';
    }
}
```

Validation asks whether input is acceptable for the field’s rules. Do not assume a text field is safe just because the browser displayed a restricted input control.


## 2. Password handling (PDF pages 6–13)

### Store a password hash, not the original password

A website should not store a user's plain-text password. A password-hashing function produces a one-way verifier. When the user logs in, the program checks the submitted password against the stored verifier.

The tutorial’s modern PHP API is `password_hash()` with `password_verify()`:

```php
$hash = password_hash($password, PASSWORD_DEFAULT); // store $hash in the DB

if (password_verify($submittedPassword, $hash)) {
    // password matches
}
```

`password_hash()` manages a random salt and stores algorithm information with the resulting string. Allocate a column large enough for future hash formats (PHP commonly recommends `VARCHAR(255)`). Do not manually salt or invent a password-hashing scheme.

### Important correction to older slide examples

The slides show MySQL `SHA()`/SHA1 as examples (PDF pages 9–10). SHA-1 is a general-purpose hash, not a suitable password-storage function today. Use PHP `password_hash()` / `password_verify()` or an established password-hashing API supported by your chosen stack. Do not store a plain password or a plain SHA/SHA1 digest.

### Transport and the password field

`<input type="password">` hides characters on screen; it does **not** encrypt the submitted value. Use HTTPS to protect credentials while they travel between browser and server.


## 3. Keep database credentials out of public code (PDF page 14)

Do not hard-code database passwords in a public PHP script or commit them to Git. Put configuration outside the web-served directory and out of version control, then load it securely into the application. Restrict access to the credential file. For deployed projects, follow the platform’s secret-management guidance.

If credentials have already been committed to a repository, removing them from the latest file is not enough: rotate/change the credentials because old versions may remain in Git history.


## 4. SQL injection and the safe way to query (PDF pages 15–24)

SQL injection happens when a program combines untrusted input with SQL text so the input can change the meaning of the query. The slides demonstrate destructive and data-stealing examples. The core problem is treating user data as part of SQL code.

### Avoid building SQL with string concatenation

```php
// Unsafe pattern: do not construct a query this way.
$sql = "SELECT email FROM Users WHERE user_id = " . $_GET['id'];
```

Input may contain SQL syntax instead of the intended ID. Validating that an ID is an integer is useful, but input validation alone is not the main SQL-injection defense.

### Use a prepared statement

```php
$stmt = $connection->prepare(
    'SELECT email FROM Users WHERE user_id = ?'
);
$stmt->bind_param('i', $id); // i means integer for MySQLi
$stmt->execute();
$result = $stmt->get_result();
```

The SQL structure is sent separately from the parameter value, so the value is treated as data. Use placeholders for user-supplied values in `SELECT`, `INSERT`, `UPDATE`, and `DELETE` statements. Table or column names cannot usually be bound as value parameters; if they must vary, select them from a strict allowlist.

The slides discuss escaping quotes with `mysqli_real_escape_string()`. Escaping can be context-sensitive and is easy to misuse. Prefer prepared statements as the normal defense. Continue validating input for correctness and business rules.


## 5. HTML / JavaScript injection (XSS) (PDF pages 25–26)

A user can submit text containing HTML or JavaScript. If the application later inserts that text directly into a page, the browser may interpret it as markup or execute code. This is commonly called **cross-site scripting (XSS)**.

Escape untrusted text for the HTML context when displaying it:

```php
echo htmlspecialchars($comment, ENT_QUOTES | ENT_SUBSTITUTE, 'UTF-8');
```

This converts special characters such as `<` and `>` so the browser displays them as text. Escape **at output time** for the output context. Input validation and output escaping solve different problems; use both where appropriate.


## 6. Server logs (PDF pages 27–29)

Servers record activity in log files. In the tutorial’s XAMPP setup, Apache has access and error logs, and MySQL can also log events.

- An **access log** records requests (often including source address, timestamp, HTTP method/path, status code, and response size).
- An **error log** helps diagnose server or application failures.
- Reviewing logs can show which pages are used, repeated failures, or unusual request patterns.

Logs are useful for troubleshooting and detecting suspicious activity. Avoid writing passwords, session tokens, or other secrets into logs. A log is evidence to investigate, not proof by itself that an attack succeeded.


## 7. Apply the checks to a form workflow

For a registration form, a safer sequence is:

1. Check required fields were submitted.
2. Trim text and validate required formats and business rules on the server.
3. Hash the password with `password_hash()`.
4. Insert the account using a prepared statement.
5. When showing later user-generated text, escape it with `htmlspecialchars()`.
6. Use HTTPS in deployment and keep DB credentials outside public code/version control.
7. Check server logs when something fails or traffic looks unusual.

These are separate protections; no single check replaces the others.


## 8. Practice questions

<details><summary>1. The HTML form requires an email address. Is that enough?</summary>

No. A request can bypass or alter the form. Validate the email on the server too.
</details>

<details><summary>2. Should passwords be stored as SHA1 values?</summary>

No. Use a password-specific API such as PHP `password_hash()` and verify with `password_verify()`. The SHA/SHA1 slides are older examples.
</details>

<details><summary>3. What is wrong with putting `$_GET['id']` directly into SQL?</summary>

The input may alter the query’s SQL syntax. Use a prepared statement with a bound parameter; validate the ID separately.
</details>

<details><summary>4. Does an HTML password input encrypt the password?</summary>

No. It only masks the characters on screen. HTTPS protects data in transit.
</details>

<details><summary>5. What does `htmlspecialchars()` protect against here?</summary>

It helps prevent submitted text from being interpreted as HTML/JavaScript when rendered in an HTML page. Use it at the output point and with the correct context.
</details>

<details><summary>6. Give two useful purposes of server logs.</summary>

Troubleshoot errors and review access patterns for unusual activity. Keep secrets out of log messages.
</details>


## Lab 5 checklist

- [ ] I know why server-side validation is required even if the browser form validates.
- [ ] I can distinguish validation from sanitizing/escaping output.
- [ ] I know why password hashes are stored and can describe `password_hash()` / `password_verify()`.
- [ ] I can explain why a password input field is not encryption and why HTTPS matters.
- [ ] I can identify SQL built by concatenating user input and replace it with a prepared statement.
- [ ] I can explain how unescaped user comments can cause XSS.
- [ ] I know what access/error logs help us investigate and what should not be written to them.

**Questions for the tutor:**  
_Write down anything you want to practise again._
